# FV Regression Workflow (New Notebook)
This notebook is independent from class examples and builds a complete FV regression pipeline.

Note: The available project CSV does not contain an explicit `FV` column, so we define `FV` from `Close` as the price value to estimate.

## 1) Setup and Imports

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import PartialDependenceDisplay
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set(style='whitegrid')

## 2) Load Dataset and Inspect Feature Vectors

In [ ]:
df = pd.read_csv('../pr15_crypto.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

print('Shape:', df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False).head(15))

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Close'].dropna(), bins=40, kde=True)
plt.title('Close Distribution (used to define FV)')
plt.xlabel('Close')
plt.ylabel('Count')
plt.show()

## 3) Preprocess Features and Target

In [ ]:
df_model = df.copy()

# Remove duplicates
df_model = df_model.drop_duplicates()

# Define target FV from available price field
df_model['FV'] = df_model['Close']

target_col = 'FV'
drop_cols = ['Close']  # avoid leakage because FV is defined from Close
X = df_model.drop(columns=[target_col] + drop_cols)
y = df_model[target_col]

print('Model dataframe shape:', df_model.shape)
print('X shape:', X.shape, '| y shape:', y.shape)
display(X.head())

## 4) Train/Validation/Test Split

In [ ]:
# 60% train, 20% validation, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=SEED
    # 0.25 of 0.8 -> 0.2 overall
 )

print('Train:', X_train.shape, y_train.shape)
print('Validation:', X_val.shape, y_val.shape)
print('Test:', X_test.shape, y_test.shape)

## 5) Feature Scaling Pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## 6) Train FV Regression Model

In [ ]:
fv_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, random_state=SEED))
])

fv_model.fit(X_train, y_train)
y_val_pred = fv_model.predict(X_val)
y_test_pred = fv_model.predict(X_test)

## 7) Evaluate Regression Performance

In [ ]:
def regression_metrics(y_true, y_pred, label='set'):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f'{label} MAE:  {mae:.6f}')
    print(f'{label} MSE:  {mse:.6f}')
    print(f'{label} RMSE: {rmse:.6f}')
    print(f'{label} R2:   {r2:.6f}')
    print('-' * 30)

y_train_pred = fv_model.predict(X_train)
regression_metrics(y_train, y_train_pred, 'Train')
regression_metrics(y_val, y_val_pred, 'Validation')
regression_metrics(y_test, y_test_pred, 'Test')

## 8) Residual and Prediction Visualizations

In [ ]:
residuals_test = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_test_pred, alpha=0.4)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_title('Predicted vs Actual (Test)')
axes[0].set_xlabel('Actual FV')
axes[0].set_ylabel('Predicted FV')

sns.histplot(residuals_test, bins=40, kde=True, ax=axes[1])
axes[1].set_title('Residual Distribution (Test)')
axes[1].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(y_test_pred, residuals_test, alpha=0.35)
plt.axhline(0, color='red', linestyle='--')
plt.title('Residuals vs Predicted (Test)')
plt.xlabel('Predicted FV')
plt.ylabel('Residual')
plt.show()

## 9) Hyperparameter Tuning

In [ ]:
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])

param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [8, 16, None],
    'regressor__min_samples_split': [2, 5]
}

grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print('Best params:', grid.best_params_)

y_val_best = best_model.predict(X_val)
y_test_best = best_model.predict(X_test)

regression_metrics(y_val, y_val_best, 'Validation (Tuned RF)')
regression_metrics(y_test, y_test_best, 'Test (Tuned RF)')

### Interpretability: Feature Importance, PDP, and SHAP

In [ ]:
# Feature importance from tuned random forest
rf_est = best_model.named_steps['regressor']
pre = best_model.named_steps['preprocessor']

feature_names = pre.get_feature_names_out()
importances = pd.Series(rf_est.feature_importances_, index=feature_names).sort_values(ascending=False)

display(importances.head(15))

plt.figure(figsize=(9, 5))
importances.head(15).sort_values().plot(kind='barh')
plt.title('Top 15 Feature Importances (Tuned RF)')
plt.xlabel('Importance')
plt.show()

In [ ]:
# PDP for top 2 transformed features
top_two = importances.head(2).index.tolist()
top_two_idx = [list(feature_names).index(f) for f in top_two]

X_val_transformed = pre.transform(X_val)

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    rf_est,
    X_val_transformed,
    features=top_two_idx,
    feature_names=feature_names,
    ax=ax
 )
plt.tight_layout()
plt.show()

In [ ]:
# SHAP summary and dependence plot (if shap is installed)
try:
    import shap

    shap_sample = pre.transform(X_val.sample(min(400, len(X_val)), random_state=SEED))
    explainer = shap.TreeExplainer(rf_est)
    shap_values = explainer.shap_values(shap_sample)

    shap.summary_plot(shap_values, shap_sample, feature_names=feature_names)

    first_feature = np.argmax(np.abs(shap_values).mean(axis=0))
    shap.dependence_plot(first_feature, shap_values, shap_sample, feature_names=feature_names)
except ImportError:
    print('Install SHAP to run this section: pip install shap')

## 10) Save Model and Run Inference

In [ ]:
model_path = './fv_regression_pipeline.joblib'
joblib.dump(best_model, model_path)
print('Saved model to:', model_path)

loaded_model = joblib.load(model_path)

new_samples = X_test.head(5).copy()
new_preds = loaded_model.predict(new_samples)

inference_df = new_samples.copy()
inference_df['Predicted_FV'] = new_preds
display(inference_df.head())

## Conclusion
This notebook created a full FV regression pipeline in a separate file, evaluated baseline and tuned models, and added interpretability with feature importance, PDP, and SHAP.